# Day 054 Project: A Chat Backend With a Memory

## What You're Building

A **persistent chat backend** (`backend.py`): a FastAPI service that saves every conversation and message to a SQLite database through your `ChatStore`. Restart the server and the history is still there. Endpoints: create a conversation, list conversations, read a conversation's messages, and post a message (which saves your turn, calls Ollama with the full saved history, and saves the reply).

## Project Requirements

1. Use the provided repository functions and `ChatStore`.
2. Build a `ChatStore` on a temp file; start a conversation, append a few messages, read the history, list conversations.
3. Prove persistence: open a **second** `ChatStore` on the same file and confirm it sees the data.
4. Call `write_backend('backend.py')` to generate the service.
5. Run `_run_project_checks()`, then `uvicorn backend:app --reload`.

## Bonus Challenges

- Add a `delete_conversation(store, cid)` (cascade removes its messages).
- Add a `pinned` column with `migrate_add_column` and a `/pin` endpoint.
- Point the frontend from Day 53 at this backend so chats persist.

## Provided: Models + Repository + ChatStore

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import tempfile
from datetime import datetime
from sqlalchemy import create_engine, ForeignKey, select, inspect as sa_inspect, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Conversation(Base):
    """One chat conversation. Has many Messages (one-to-many)."""
    __tablename__ = 'conversations'

    id:         Mapped[int]      = mapped_column(primary_key=True)
    title:      Mapped[str]      = mapped_column(default='New chat')
    created_at: Mapped[datetime] = mapped_column(default=datetime.utcnow)

    # relationship() is the ORM link (not a DB column). cascade deletes a
    # conversation's messages when the conversation is deleted.
    messages: Mapped[list['Message']] = relationship(
        back_populates='conversation', cascade='all, delete-orphan')


class Message(Base):
    """One message in a conversation. Belongs to one Conversation (many-to-one)."""
    __tablename__ = 'messages'

    id:              Mapped[int]      = mapped_column(primary_key=True)
    conversation_id: Mapped[int]      = mapped_column(ForeignKey('conversations.id'))
    role:            Mapped[str]      = mapped_column()
    content:         Mapped[str]      = mapped_column()
    created_at:      Mapped[datetime] = mapped_column(default=datetime.utcnow)

    conversation: Mapped['Conversation'] = relationship(back_populates='messages')


def memory_engine():
    """In-memory SQLite engine for tests. StaticPool makes every Session share the
    one in-memory database (see Day 44)."""
    return create_engine('sqlite:///:memory:',
                          connect_args={'check_same_thread': False},
                          poolclass=StaticPool)


def create_schema(engine) -> None:
    """Create every table registered on Base (CREATE TABLE IF NOT EXISTS)."""
    Base.metadata.create_all(engine)


def create_conversation(session, title: str = 'New chat') -> Conversation:
    """Insert a new conversation and flush so its auto id is assigned.
    The caller controls commit (unit-of-work pattern)."""
    conv = Conversation(title=title)
    session.add(conv)
    session.flush()
    return conv


def add_message(session, conversation_id: int, role: str, content: str) -> Message:
    """Append a message to a conversation via its foreign key, and flush to assign
    the id. The caller commits."""
    msg = Message(conversation_id=conversation_id, role=role, content=content)
    session.add(msg)
    session.flush()
    return msg


def get_messages(session, conversation_id: int) -> list:
    """Return the conversation's messages, in insertion order, as plain
    [{'role', 'content'}] dicts (safe to use after the session closes)."""
    stmt = (select(Message)
            .where(Message.conversation_id == conversation_id)
            .order_by(Message.id))
    rows = session.execute(stmt).scalars().all()
    return [{'role': m.role, 'content': m.content} for m in rows]


def list_conversations(session) -> list:
    """Return all conversations as [{'id', 'title', 'message_count'}] by id."""
    convs = session.execute(
        select(Conversation).order_by(Conversation.id)).scalars().all()
    return [{'id': c.id, 'title': c.title, 'message_count': len(c.messages)}
            for c in convs]


def make_engine(db_path: str):
    """File-backed SQLite engine — data SURVIVES a process restart. Creates the
    schema on first use (safe to call every startup)."""
    engine = create_engine(f'sqlite:///{db_path}')
    Base.metadata.create_all(engine)
    return engine


def column_exists(engine, table: str, column: str) -> bool:
    """True if `column` already exists on `table` (schema introspection)."""
    return column in [c['name'] for c in sa_inspect(engine).get_columns(table)]


def migrate_add_column(engine, table: str, column: str, sqltype: str = 'TEXT') -> bool:
    """A minimal, idempotent migration: add a column only if it is missing.
    Returns True if it added the column, False if it was already there.
    Safe to run on every startup — this is the essence of a migration."""
    if column_exists(engine, table, column):
        return False
    with engine.begin() as conn:
        conn.execute(text(f'ALTER TABLE {table} ADD COLUMN {column} {sqltype}'))
    return True


class ChatStore:
    """Persistence facade for the chat app. One file-backed database; each method
    opens a short-lived Session and commits atomically. The app calls these
    methods and never touches SQL — the same thin-shell pattern as the earlier
    days, now over a database."""

    def __init__(self, db_path: str):
        self.engine = make_engine(db_path)

    def start(self, title: str = 'New chat') -> int:
        with Session(self.engine) as s:
            conv = create_conversation(s, title)
            s.commit()
            return conv.id

    def append(self, conversation_id: int, role: str, content: str) -> int:
        with Session(self.engine) as s:
            msg = add_message(s, conversation_id, role, content)
            s.commit()
            return msg.id

    def history(self, conversation_id: int) -> list:
        with Session(self.engine) as s:
            return get_messages(s, conversation_id)

    def conversations(self) -> list:
        with Session(self.engine) as s:
            return list_conversations(s)

## Provided: Backend File Writer

In [ ]:
from pathlib import Path

# The persistent chat backend: models + repository + ChatStore + FastAPI
# routes, embedded as a string so the notebook can write it verbatim.
_BACKEND_SRC = 'import warnings\nwarnings.filterwarnings(\'ignore\')\nfrom datetime import datetime\nfrom fastapi import FastAPI\nfrom pydantic import BaseModel, Field\nfrom sqlalchemy import create_engine, ForeignKey, select, inspect as sa_inspect, text\nfrom sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session\nimport ollama\n\n\nclass Base(DeclarativeBase):\n    pass\n\n\nclass Conversation(Base):\n    """One chat conversation. Has many Messages (one-to-many)."""\n    __tablename__ = \'conversations\'\n\n    id:         Mapped[int]      = mapped_column(primary_key=True)\n    title:      Mapped[str]      = mapped_column(default=\'New chat\')\n    created_at: Mapped[datetime] = mapped_column(default=datetime.utcnow)\n\n    # relationship() is the ORM link (not a DB column). cascade deletes a\n    # conversation\'s messages when the conversation is deleted.\n    messages: Mapped[list[\'Message\']] = relationship(\n        back_populates=\'conversation\', cascade=\'all, delete-orphan\')\n\n\nclass Message(Base):\n    """One message in a conversation. Belongs to one Conversation (many-to-one)."""\n    __tablename__ = \'messages\'\n\n    id:              Mapped[int]      = mapped_column(primary_key=True)\n    conversation_id: Mapped[int]      = mapped_column(ForeignKey(\'conversations.id\'))\n    role:            Mapped[str]      = mapped_column()\n    content:         Mapped[str]      = mapped_column()\n    created_at:      Mapped[datetime] = mapped_column(default=datetime.utcnow)\n\n    conversation: Mapped[\'Conversation\'] = relationship(back_populates=\'messages\')\n\n\ndef create_schema(engine) -> None:\n    """Create every table registered on Base (CREATE TABLE IF NOT EXISTS)."""\n    Base.metadata.create_all(engine)\n\n\ndef create_conversation(session, title: str = \'New chat\') -> Conversation:\n    """Insert a new conversation and flush so its auto id is assigned.\n    The caller controls commit (unit-of-work pattern)."""\n    conv = Conversation(title=title)\n    session.add(conv)\n    session.flush()\n    return conv\n\n\ndef add_message(session, conversation_id: int, role: str, content: str) -> Message:\n    """Append a message to a conversation via its foreign key, and flush to assign\n    the id. The caller commits."""\n    msg = Message(conversation_id=conversation_id, role=role, content=content)\n    session.add(msg)\n    session.flush()\n    return msg\n\n\ndef get_messages(session, conversation_id: int) -> list:\n    """Return the conversation\'s messages, in insertion order, as plain\n    [{\'role\', \'content\'}] dicts (safe to use after the session closes)."""\n    stmt = (select(Message)\n            .where(Message.conversation_id == conversation_id)\n            .order_by(Message.id))\n    rows = session.execute(stmt).scalars().all()\n    return [{\'role\': m.role, \'content\': m.content} for m in rows]\n\n\ndef list_conversations(session) -> list:\n    """Return all conversations as [{\'id\', \'title\', \'message_count\'}] by id."""\n    convs = session.execute(\n        select(Conversation).order_by(Conversation.id)).scalars().all()\n    return [{\'id\': c.id, \'title\': c.title, \'message_count\': len(c.messages)}\n            for c in convs]\n\n\ndef make_engine(db_path: str):\n    """File-backed SQLite engine — data SURVIVES a process restart. Creates the\n    schema on first use (safe to call every startup)."""\n    engine = create_engine(f\'sqlite:///{db_path}\')\n    Base.metadata.create_all(engine)\n    return engine\n\n\ndef column_exists(engine, table: str, column: str) -> bool:\n    """True if `column` already exists on `table` (schema introspection)."""\n    return column in [c[\'name\'] for c in sa_inspect(engine).get_columns(table)]\n\n\ndef migrate_add_column(engine, table: str, column: str, sqltype: str = \'TEXT\') -> bool:\n    """A minimal, idempotent migration: add a column only if it is missing.\n    Returns True if it added the column, False if it was already there.\n    Safe to run on every startup — this is the essence of a migration."""\n    if column_exists(engine, table, column):\n        return False\n    with engine.begin() as conn:\n        conn.execute(text(f\'ALTER TABLE {table} ADD COLUMN {column} {sqltype}\'))\n    return True\n\n\nclass ChatStore:\n    """Persistence facade for the chat app. One file-backed database; each method\n    opens a short-lived Session and commits atomically. The app calls these\n    methods and never touches SQL — the same thin-shell pattern as the earlier\n    days, now over a database."""\n\n    def __init__(self, db_path: str):\n        self.engine = make_engine(db_path)\n\n    def start(self, title: str = \'New chat\') -> int:\n        with Session(self.engine) as s:\n            conv = create_conversation(s, title)\n            s.commit()\n            return conv.id\n\n    def append(self, conversation_id: int, role: str, content: str) -> int:\n        with Session(self.engine) as s:\n            msg = add_message(s, conversation_id, role, content)\n            s.commit()\n            return msg.id\n\n    def history(self, conversation_id: int) -> list:\n        with Session(self.engine) as s:\n            return get_messages(s, conversation_id)\n\n    def conversations(self) -> list:\n        with Session(self.engine) as s:\n            return list_conversations(s)\n\n\nclass ConversationIn(BaseModel):\n    title: str = \'New chat\'\n\n\nclass MessageIn(BaseModel):\n    message: str = Field(min_length=1)\n\n\nstore = ChatStore(\'chat.db\')\napp = FastAPI(title=\'Persistent Chat API\')\n\n\n@app.post(\'/conversations\')\ndef create_conv(req: ConversationIn):\n    return {\'id\': store.start(req.title)}\n\n\n@app.get(\'/conversations\')\ndef list_convs():\n    return {\'conversations\': store.conversations()}\n\n\n@app.get(\'/conversations/{cid}/messages\')\ndef get_history(cid: int):\n    return {\'messages\': store.history(cid)}\n\n\n@app.post(\'/conversations/{cid}/messages\')\ndef post_message(cid: int, req: MessageIn):\n    store.append(cid, \'user\', req.message)\n    history = store.history(cid)          # full saved history each turn\n    try:\n        resp = ollama.chat(model=\'llama3.2\', messages=history)\n        reply = resp[\'message\'][\'content\'].strip()\n    except Exception as e:\n        reply = f\'[Model unavailable: {e}]\'\n    store.append(cid, \'assistant\', reply)\n    return {\'reply\': reply}\n\n\nif __name__ == \'__main__\':\n    import uvicorn\n    uvicorn.run(app, host=\'0.0.0.0\', port=8000)\n'


def write_backend(path: str = 'backend.py') -> str:
    """Write the persistent chat backend to `path` and return the path."""
    Path(path).write_text(_BACKEND_SRC, encoding='utf-8')
    return path

## Your Pipeline

In [ ]:
# TODO: db_path = os.path.join(tempfile.mkdtemp(), 'chat.db')
# TODO: store = ChatStore(db_path)
# TODO: cid = store.start('My first saved chat')
# TODO: store.append(cid, 'user', 'Remember this message.')
# TODO: store.append(cid, 'assistant', 'Saved!')
# TODO: print('history     :', store.history(cid))
# TODO: print('conversations:', store.conversations())
#
# TODO: reopened = ChatStore(db_path)   # simulate a restart
# TODO: print('after restart:', reopened.history(cid))
#
# TODO: print('wrote', write_backend('backend.py'))
# TODO: print('Run:  uvicorn backend:app --reload')

## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: a ChatStore was created and used
    try:
        assert 'store' in globals() and isinstance(store, ChatStore), 'create store = ChatStore(...)'
        assert 'cid' in globals(), 'start a conversation and keep its id in cid'
        passed += 1; print('✅ Check 1: ChatStore created, conversation started')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: history has the saved turns
    try:
        h = store.history(cid)
        assert len(h) >= 2, f'expected >= 2 saved messages, got {len(h)}'
        passed += 1; print(f'✅ Check 2: {len(h)} messages saved')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: persistence across a fresh ChatStore
    try:
        assert 'reopened' in globals() and isinstance(reopened, ChatStore), 'open a second ChatStore'
        assert reopened.history(cid) == store.history(cid), 'reopened store must see the same data'
        passed += 1; print('✅ Check 3: data persists across a restart')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: backend.py generated + valid Python
    try:
        assert os.path.exists('backend.py'), 'backend.py not found — call write_backend()'
        src = open('backend.py', encoding='utf-8').read()
        assert 'ChatStore' in src and 'from fastapi import FastAPI' in src
        compile(src, 'backend.py', 'exec')
        passed += 1; print('✅ Check 4: backend.py generated and compiles')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: backend persists via ChatStore + has the routes
    try:
        src = open('backend.py', encoding='utf-8').read()
        assert "store = ChatStore(" in src, 'backend must instantiate a ChatStore'
        assert "@app.post('/conversations')" in src and "@app.get('/conversations')" in src
        passed += 1; print('✅ Check 5: backend wires ChatStore into routes')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Project complete! Run: uvicorn backend:app --reload')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()